# Phase 3A (Kaggle): merge the adapter, then make GGUF versions

**What is happening, in plain words**
- The Phase 2 *adapter* is a small add-on file. **Merging** folds it into the full model, giving one normal model
  (about 9 GB in 16-bit "fp16"). Everything below starts from that.
- **Quantization** shrinks a model by storing each weight with fewer bits (16 -> 4 or 8). Smaller files, less memory,
  faster serving, at the cost of a little accuracy. This phase *measures* that cost.
- **GGUF** is a compact model file format. **llama.cpp** is a program that runs GGUF models efficiently, on laptops,
  CPUs or GPUs. We build it here to convert the model and to serve it for testing.
- We then score each version on the same 3,080 test examples and compare size vs accuracy.

**Steps:** merge -> score the merged fp16 model (checks the merge itself caused no damage) -> convert to GGUF ->
make Q8_0 (8-bit) and Q4_K_M (4-bit) -> serve each with llama.cpp -> score -> table.

**Before you run**
1. Upload the newest `banking-finetuning-code.zip` as a new version of your code dataset (it has the Phase 3 code).
2. Make a Kaggle Dataset from your Phase 2 adapter folder (the one containing `adapter_model.safetensors`) and add it here.
3. Accelerator = **GPU T4 x2**, Internet = **On**. Optional: add a Kaggle Secret `HF_TOKEN` to back up big files privately.

**Time:** about 1.5 hours. Kaggle wipes `/kaggle/working` when the session ends: click each download link as it appears,
or use Save Version -> Save & Run All.

In [ ]:
import os, glob, shutil, subprocess, sys, json, time, threading

def find_src():
    hits = glob.glob('/kaggle/input/**/src/schema.py', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        shutil.unpack_archive(z, '/kaggle/working/_code')
        hits = glob.glob('/kaggle/working/_code/**/src/schema.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    print(subprocess.run('find /kaggle/input -maxdepth 4 | head -40', shell=True, capture_output=True, text=True).stdout)
    raise FileNotFoundError('code dataset not found: add banking-finetuning-code via Add Input')

SRC_ROOT = find_src()
PROJ = '/kaggle/working/Banking_FInetuning'
shutil.copytree(SRC_ROOT, PROJ, ignore=shutil.ignore_patterns('__pycache__'), dirs_exist_ok=True)
os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
assert os.path.exists('configs/phase3/fp16.yaml'), 'Old code dataset: upload the newest banking-finetuning-code.zip as a new version'

# Big files (merged model, GGUF, AWQ) go to /kaggle/tmp (lots of space); /kaggle/working is capped at ~20 GB.
os.makedirs('/kaggle/tmp/artifacts', exist_ok=True)
if not os.path.exists('artifacts'):
    os.symlink('/kaggle/tmp/artifacts', 'artifacts')

# The trained adapter from Phase 2: expected as a Kaggle dataset (folder containing adapter_model.safetensors).
ad = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
assert ad, 'Add your adapter dataset via Add Input (the Phase 2 adapter folder, containing adapter_model.safetensors)'
ADAPTER_DST = 'outputs/full/qwen3_5_4b/adapter'
shutil.copytree(os.path.dirname(ad[0]), ADAPTER_DST, dirs_exist_ok=True)
print('project at', PROJ, '| adapter from', os.path.dirname(ad[0]), '->', ADAPTER_DST)

In [ ]:
!pip list --format=freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt
!cat /tmp/constraints.txt
!pip install -q -U -c /tmp/constraints.txt transformers peft accelerate scikit-learn pyyaml pandas datasets requests sentencepiece
# Kaggle ships torchao 0.10.0, which the newest peft rejects when merging into a non-quantized model. We do not use torchao.
!pip uninstall -y -q torchao

In [ ]:
import torch, transformers, peft
print('torch', torch.__version__, '| built for CUDA:', torch.version.cuda, '| cuda available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
print('transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.version.cuda is not None and torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2 and restart the session'
print(subprocess.run('nvcc --version | tail -2', shell=True, capture_output=True, text=True).stdout or 'nvcc NOT found (needed to build llama.cpp for GPU)')

In [ ]:
from IPython.display import FileLink, display

def stream(cmd, log_path=None, env=None):
    # Run a command, stream its output into the notebook (and a log file), raise on failure.
    print('$', ' '.join(map(str, cmd)))
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    log = open(log_path, 'a') if log_path else None
    for line in p.stdout:
        if 'it/s' in line or 's/it' in line:
            continue
        print(line, end='')
        if log:
            log.write(line); log.flush()
    rc = p.wait()
    if rc:
        raise RuntimeError(f'command failed (exit {rc}); see output above')

def save_stage(zip_name, rel_paths):
    # Zip files (paths relative to the project) into /kaggle/working and show a download link.
    here = os.getcwd()
    os.chdir('/kaggle/working')
    try:
        paths = ' '.join(f'Banking_FInetuning/{p}' for p in rel_paths if os.path.exists(f'/kaggle/working/Banking_FInetuning/{p}'))
        subprocess.run(f'rm -f {zip_name}.zip && zip -qr {zip_name}.zip {paths}', shell=True, check=True)
        print(f'{zip_name}.zip = {os.path.getsize(zip_name + ".zip") / 1e6:.1f} MB  <- CLICK TO DOWNLOAD NOW')
        display(FileLink(f'{zip_name}.zip'))
    finally:
        os.chdir(here)

def push_to_hub(local_path, repo_name, path_in_repo='', ignore=None):
    # Optional private upload to YOUR Hugging Face account so big artifacts survive the session and can be
    # pulled into Colab later. Needs a Kaggle Secret named HF_TOKEN (write access). Skips quietly if absent.
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        print('No HF_TOKEN secret found: skipping the Hugging Face upload.')
        return
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/{repo_name}"
    api.create_repo(repo_id, private=True, exist_ok=True)
    if os.path.isdir(local_path):
        api.upload_folder(folder_path=local_path, repo_id=repo_id, path_in_repo=path_in_repo, ignore_patterns=ignore)
    else:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo or os.path.basename(local_path), repo_id=repo_id)
    print('uploaded (private) ->', f'https://huggingface.co/{repo_id}')

def du(path):
    p = subprocess.run(['du', '-shL', path], capture_output=True, text=True)
    return p.stdout.split()[0] if p.stdout else '?'

def run_jobs(jobs):
    # jobs: {name: (gpu, [(module, [args...]), ...])}. Each job runs in its own thread on its own GPU.
    results = {}
    def worker(name, gpu, steps):
        with open(f'logs/{name}.log', 'w') as log:
            for module, args in steps:
                print('$ python -m', module, *map(str, args), file=log, flush=True)
                p = subprocess.run([sys.executable, '-m', module, *map(str, args)], stdout=log, stderr=subprocess.STDOUT,
                                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
                if p.returncode:
                    results[name] = f'FAILED in {module} (exit {p.returncode})'
                    return
        results[name] = 'ok'
    def last_lines(name):
        f = f'logs/{name}.log'
        L = [l for l in open(f).read().splitlines() if l.strip() and 'it/s' not in l and 's/it' not in l] if os.path.exists(f) else []
        return ' | '.join(l[:140] for l in L[-2:])
    ts = [threading.Thread(target=worker, args=(n, g, s)) for n, (g, s) in jobs.items()]
    [t.start() for t in ts]
    while any(t.is_alive() for t in ts):
        time.sleep(60)
        for n in jobs:
            print(f'[{n}]', last_lines(n))
        print('---')
    [t.join() for t in ts]
    print(results)
    for n, r in results.items():
        if r != 'ok':
            print(f'===== {n}: {r} - last log lines =====')
            print(subprocess.run(f"grep -v -E 'it/s|s/it' logs/{n}.log | tail -45", shell=True, capture_output=True, text=True).stdout)
    return results

## 1. Data
Only the test split is needed for scoring (3,080 examples, never used in training).

In [ ]:
if not os.path.exists('data/test.jsonl'):
    stream([sys.executable, '-m', 'src.data.prepare'])
print(subprocess.run('wc -l data/test.jsonl', shell=True, capture_output=True, text=True).stdout)

## 2. Merge the adapter into the base model
Downloads the base Qwen3.5-4B, folds the adapter in, saves one fp16 model to `artifacts/merged-fp16`. About 5 minutes.

In [ ]:
stream([sys.executable, '-m', 'src.quantize.merge', '--adapter', ADAPTER_DST, '--out', 'artifacts/merged-fp16'], log_path='logs/merge.log')
print('merged model size:', du('artifacts/merged-fp16'))

## 3. Score the merged fp16 model (merge sanity check)
This is the reference for every compressed version. It should match Phase 2's score (intent accuracy **0.9393**,
macro-F1 **0.9398**, 4 invented labels). A small difference is normal because the adapter was trained on a 4-bit copy
of the model but is merged into the 16-bit one; a big drop would mean the merge itself is a problem. About 20 min.

In [ ]:
stream([sys.executable, '-m', 'src.eval.hf_eval', '--config', 'configs/phase3/fp16.yaml', '--split', 'data/test.jsonl',
        '--tag', 'finetuned_test', '--batch-size', 16], log_path='logs/eval_fp16.log', env={'CUDA_VISIBLE_DEVICES': '0'})
m = json.load(open('outputs/phase3/fp16/finetuned_test.json'))
P2 = {'intent_accuracy': 0.9393, 'intent_macro_f1': 0.9398, 'invented_label_rate': 0.0013, 'urgency_accuracy': 0.9828, 'needs_human_accuracy': 0.9919}
print(f"{'metric':<24}{'Phase 2':>10}{'merged fp16':>14}{'change':>10}")
for k, v in P2.items():
    print(f'{k:<24}{v:>10.4f}{m[k]:>14.4f}{m[k] - v:>+10.4f}')

In [ ]:
save_stage('phase3a_fp16', ['outputs/phase3/fp16', 'logs'])

## 4. Build llama.cpp
**llama.cpp** is an open-source program that runs language models in the GGUF format. It has two tools we need:
`llama-quantize` (shrinks a GGUF) and `llama-server` (serves a GGUF over a web API so we can score it).
We compile it for the T4 GPU (architecture 75). It takes about 10 minutes.

In [ ]:
LLAMA = '/kaggle/tmp/llama.cpp'
if not os.path.exists(LLAMA):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', LLAMA], check=True)
!pip install -q -c /tmp/constraints.txt /kaggle/tmp/llama.cpp/gguf-py sentencepiece

In [ ]:
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

# CMake needs a file literally named libcuda.so to link against. This machine has the GPU driver's copy under a
# versioned name (libcuda.so.1) but not the development link, so we make a small symlink and point CMake at it.
cands = (glob.glob('/usr/local/cuda*/lib64/stubs/libcuda.so') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
         + glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/local/nvidia/lib64/libcuda.so*'))
print('libcuda candidates:', cands)
assert cands, 'No libcuda found anywhere: copy this message to Claude'
STUB = '/kaggle/tmp/cudastub'
os.makedirs(STUB, exist_ok=True)
if os.path.lexists(f'{STUB}/libcuda.so'):
    os.remove(f'{STUB}/libcuda.so')
os.symlink(cands[0], f'{STUB}/libcuda.so')
shutil.rmtree('/kaggle/tmp/llama.cpp/build', ignore_errors=True)    # start the configure from a clean state
cfg = (f'cmake -S /kaggle/tmp/llama.cpp -B /kaggle/tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 '
       f'-DCUDAToolkit_ROOT=/usr/local/cuda -DCMAKE_LIBRARY_PATH={STUB} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release '
       f'> /kaggle/tmp/cmake.log 2>&1')
rc = subprocess.run(cfg, shell=True).returncode
print('cmake configure exit code:', rc)
if rc:
    print('--- the real errors (not just the last line) ---')
    print(sh("grep -n -i -B2 -A6 -E 'CMake Error|could not|not found|missing' /kaggle/tmp/cmake.log | head -80"))
    print('--- CUDA compiler ---'); print(sh('which nvcc; nvcc --version | tail -2'))
    raise RuntimeError('llama.cpp CUDA configure failed: copy the errors above to Claude')
rc = subprocess.run('cmake --build /kaggle/tmp/llama.cpp/build --target llama-server llama-quantize -j4 > /kaggle/tmp/build.log 2>&1', shell=True).returncode
print('build exit code:', rc)
if rc:
    print(sh("grep -n -i -B3 -A3 -E 'error' /kaggle/tmp/build.log | head -60"))
    raise RuntimeError('llama.cpp build failed: copy the errors above to Claude')
LLAMA_BIN = '/kaggle/tmp/llama.cpp/build/bin'
assert os.path.exists(f'{LLAMA_BIN}/llama-server') and os.path.exists(f'{LLAMA_BIN}/llama-quantize')
print('built:', os.listdir(LLAMA_BIN)[:8])

## 5. Convert to GGUF and quantize
1. Convert the merged model to a full-precision GGUF (`f16`, about 9 GB).
2. `Q8_0` = 8 bits per weight (about half the size, almost lossless).
3. `Q4_K_M` = about 4.5 bits per weight (about a quarter the size; a popular balance of size and quality).

In [ ]:
# The merged model has no MTP head (an optional speculative-decoding extra), so tell the converter not to expect one.
# Without --no-mtp the file declares 33 layers but contains 32 and llama.cpp refuses to load it ("blk.32 not found").
for f in glob.glob('artifacts/gguf/*.gguf'):
    os.remove(f)
os.makedirs('artifacts/gguf', exist_ok=True)
stream([sys.executable, f'{LLAMA}/convert_hf_to_gguf.py', 'artifacts/merged-fp16', '--outfile', 'artifacts/gguf/model-f16.gguf',
        '--outtype', 'f16', '--no-mtp'], log_path='logs/gguf_convert.log')
for q in ['Q8_0', 'Q4_K_M']:
    stream([f'{LLAMA_BIN}/llama-quantize', 'artifacts/gguf/model-f16.gguf', f'artifacts/gguf/model-{q}.gguf', q], log_path='logs/gguf_quant.log')
print(subprocess.run('ls -lh artifacts/gguf', shell=True, capture_output=True, text=True).stdout)

## 6. Score each GGUF
We start `llama-server` (a local web server) for one model at a time, send it the 3,080 test prompts (the exact same
prompt text used in training, so the comparison is fair), and score the answers. First a 3-example peek to make sure the
output looks like JSON, then the full run (about 10-15 min each).

In [ ]:
import requests
def start_server(gguf, port=8000):
    log = open('logs/llama_server.log', 'w')
    p = subprocess.Popen([f'{LLAMA_BIN}/llama-server', '-m', gguf, '--alias', 'banking', '--host', '127.0.0.1', '--port', str(port),
                          '-ngl', '99', '-c', '8192', '-np', '8'], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(240):
        try:
            if requests.get(f'http://127.0.0.1:{port}/health', timeout=2).status_code == 200:
                return p
        except Exception:
            pass
        if p.poll() is not None:
            raise RuntimeError('llama-server exited: ' + subprocess.run('tail -30 logs/llama_server.log', shell=True, capture_output=True, text=True).stdout)
        time.sleep(2)
    raise RuntimeError('llama-server did not become ready')

def score_gguf(q):
    name = f'gguf_{q.lower()}'
    d = f'outputs/phase3/{name}'
    os.makedirs(d, exist_ok=True)
    p = start_server(f'artifacts/gguf/model-{q}.gguf')
    try:
        stream([sys.executable, '-m', 'src.eval.evaluate', '--model', 'banking', '--tokenizer', 'artifacts/merged-fp16', '--limit', 3,
                '--out', f'{d}/peek.json'])
        pk = json.load(open(f'{d}/peek.json'))
        assert pk['json_valid_rate'] > 0, 'The server returned no valid JSON on the 3-example peek (special tokens or template problem). Stopping before the long run.'
        stream([sys.executable, '-m', 'src.eval.evaluate', '--model', 'banking', '--tokenizer', 'artifacts/merged-fp16', '--concurrency', 8,
                '--out', f'{d}/finetuned_test.json', '--preds-out', f'{d}/finetuned_test_preds.jsonl', '--tag', name], log_path=f'logs/eval_{name}.log')
    finally:
        p.terminate(); p.wait(); time.sleep(5)

In [ ]:
score_gguf('Q8_0')

In [ ]:
score_gguf('Q4_K_M')

## 7. Comparison table
Quality vs size, all on the same 3,080 test examples. **same answer as fp16** = how often the quantized model gives exactly the same intent as the 16-bit one.

In [ ]:
stream([sys.executable, '-m', 'src.eval.quant_table',
        '--variant', 'fp16', 'outputs/phase3/fp16', 'artifacts/merged-fp16',
        '--variant', 'gguf Q8_0', 'outputs/phase3/gguf_q8_0', 'artifacts/gguf/model-Q8_0.gguf',
        '--variant', 'gguf Q4_K_M', 'outputs/phase3/gguf_q4_k_m', 'artifacts/gguf/model-Q4_K_M.gguf',
        '--out', 'results/phase3/quant_table_gguf.md'])

## 8. Save now
Download the zip. The optional upload puts the GGUF files and the merged model in a private Hugging Face repo, so Phase 4 (on Colab) can pull them.

In [ ]:
save_stage('phase3a_results', ['outputs/phase3', 'results/phase3', 'logs'])

In [ ]:
push_to_hub('artifacts/gguf', 'banking-triage-qwen3_5_4b-quantized', path_in_repo='gguf', ignore=['*f16*'])
push_to_hub('artifacts/merged-fp16', 'banking-triage-qwen3_5_4b-quantized', path_in_repo='merged-fp16')